# 05_build_universe_matrix

**Objective:** Convert the prior-art universe into a memory-mapped, L2-normalised float16 embedding matrix and a compact metadata table for fast novelty computation.

**Inputs:** `prior_art_universe.parquet`.

**Outputs:** `universe_emb_norm.f16.bin` (N x 1024 float16); `universe_meta.parquet`.

In [ ]:
# Imports
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm import tqdm

## Paths and configuration

In [ ]:
# Paths and configuration
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
SRC      = PROC / "prior_art_universe.parquet"
EMB_BIN  = PROC / "universe_emb_norm.f16.bin"
META     = PROC / "universe_meta.parquet"

DIM   = 1024
BATCH = 50_000

In [ ]:
# Open source parquet and report sizes
t0 = time.time()
print(f"=== Schritt 6: Universe -> Memmap ===")

pf = pq.ParquetFile(SRC)
N = pf.metadata.num_rows
print(f"  rows: {N:,}, dim: {DIM}")
print(f"  memmap size: {N * DIM * 2 / 1e9:.2f} GB")

In [ ]:
# Allocate output memmap
print(f"\nAllokiere memmap: {EMB_BIN.name}")
emb_mm = np.memmap(EMB_BIN, dtype=np.float16, mode="w+", shape=(N, DIM))

In [ ]:
# Allocate metadata arrays
fam_ids   = np.full(N, -1, dtype=np.int64)
priority  = np.full(N, np.iinfo(np.int32).min, dtype=np.int32)
src_codes = np.empty(N, dtype=np.int8)
lens_buf  = [None] * N
SRC_MAP = {"EPO": 0, "USPTO": 1, "RECOVERED": 2}
EPOCH_DAY = pd.Timestamp("1970-01-01").toordinal()

## Stream batches, normalise embeddings, collect metadata

In [ ]:
# Stream batches, normalise embeddings, and collect metadata
idx = 0
zero_norm = 0
print(f"\nStreame und normalisiere ...")
pbar = tqdm(total=N, unit="rows")
for batch in pf.iter_batches(
        batch_size=BATCH,
        columns=["embedding","priority_date","docdb_family_id","lens_id","embedding_source"]):
    n = len(batch)

    emb_flat = batch.column("embedding").values.to_numpy(zero_copy_only=False)
    emb_2d_f32 = emb_flat.reshape(n, DIM).astype(np.float32)
    norms = np.linalg.norm(emb_2d_f32, axis=1, keepdims=True)
    bad = (norms.squeeze() < 1e-8)
    if bad.any():
        zero_norm += int(bad.sum())
        norms[bad] = 1.0
    emb_norm_f16 = (emb_2d_f32 / norms).astype(np.float16)
    emb_mm[idx:idx+n] = emb_norm_f16

    fam_arr = batch.column("docdb_family_id").to_pandas()
    fam_ids[idx:idx+n] = fam_arr.fillna(-1).astype("int64").values

    pd_arr = batch.column("priority_date").to_pandas()
    valid = pd_arr.notna().values
    if valid.any():
        days = (pd.to_datetime(pd_arr[valid]).map(lambda x: x.toordinal() - EPOCH_DAY)
                .astype("int32").values)
        idxs_valid = np.where(valid)[0] + idx
        priority[idxs_valid] = days

    src_arr = batch.column("embedding_source").to_pylist()
    for i, s in enumerate(src_arr):
        src_codes[idx+i] = SRC_MAP[s]

    lens_arr = batch.column("lens_id").to_pylist()
    for i, lid in enumerate(lens_arr):
        if lid is not None:
            lens_buf[idx+i] = lid

    idx += n
    pbar.update(n)
pbar.close()

In [ ]:
# Flush and close the memmap
emb_mm.flush()
del emb_mm
print(f"\n  embeddings written: {idx:,}")
print(f"  zero-norm patents : {zero_norm}")

## Write compact metadata parquet

In [ ]:
# Write the compact metadata parquet
print(f"\nSchreibe Metadata: {META.name}")
meta_df = pd.DataFrame({
    "row_idx":           np.arange(N, dtype=np.int32),
    "docdb_family_id":   pd.array(np.where(fam_ids == -1, pd.NA, fam_ids), dtype="Int64"),
    "lens_id":           pd.array(lens_buf, dtype="string"),
    "priority_date_days":priority,
    "embedding_source":  pd.Categorical.from_codes(src_codes, categories=["EPO","USPTO","RECOVERED"]),
})
meta_df.to_parquet(META, compression="zstd")
print(f"  -> {META} ({META.stat().st_size/1e6:.1f} MB)")

## Sanity check normalisation and runtime

In [ ]:
# Sanity-check normalisation and report runtime
print(f"\n=== Sanity ===")
emb_check = np.memmap(EMB_BIN, dtype=np.float16, mode="r", shape=(N, DIM))
sample_idx = [0, N//2, N-1]
for i in sample_idx:
    v = emb_check[i].astype(np.float32)
    print(f"  row {i:>10,}: norm={np.linalg.norm(v):.4f}, "
          f"src={meta_df['embedding_source'].iloc[i]}, "
          f"date_days={meta_df['priority_date_days'].iloc[i]}")

print(f"\n  total runtime: {(time.time()-t0)/60:.1f} min")
print(f"  memmap : {EMB_BIN}  ({EMB_BIN.stat().st_size/1e9:.2f} GB)")
print(f"  meta   : {META}     ({META.stat().st_size/1e6:.1f} MB)")